In [18]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms as trn

import clip
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

from data_loading import load_sun_data
from dataloaders import create_sun_dataloader
from cav_utils import calculate_filtered_cav, get_cosine_similarity, get_dot_product_similarity, calculate_centroid_cav

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Używane urządzenie: {DEVICE}")

Używane urządzenie: cuda


In [19]:
model_file = 'models/resnet18_places365.pth'
conv_model = models.resnet18(num_classes=365)
checkpoint = torch.load(model_file, map_location=lambda storage, loc: storage)
state_dict = {str.replace(k, 'module.', ''): v for k, v in checkpoint['state_dict'].items()}
conv_model.load_state_dict(state_dict)
conv_model.fc = nn.Identity() 
conv_model = conv_model.to(DEVICE).eval()

centre_crop = trn.Compose([
    trn.Resize((256, 256)),
    trn.CenterCrop(224),
    trn.ToTensor(),
    trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

clip_model, clip_preprocess = clip.load('ViT-B/32', device=DEVICE)
clip_model.eval()
print(f'Modele załadowane. Wymiar embeddingu CLIP: {clip_model.visual.output_dim}')

Modele załadowane. Wymiar embeddingu CLIP: 512


In [20]:
df_sun, attributes_list, attr_cols = load_sun_data(base_path='data/SUN/SUNAttributeDB', threshold=0.5)

print(f'Images read: {len(df_sun)}')
print(f'Number of attributes: {len(attr_cols)}')

df_sun.head(3)

Images read: 14340
Number of attributes: 102


,image_id,file_path,class_id,sailing/ boating,driving,biking,transporting things or people,sunbathing,vacationing/ touring,hiking,...,far-away horizon,no horizon,rugged scene,mostly vertical components,mostly horizontal components,symmetrical,cluttered space,scary,soothing,stressful
0,1,a/abbey/sun_aakbdcgfpksytcwj.jpg,a,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1,2,a/abbey/sun_aaoktempcmudsvna.jpg,a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,a/abbey/sun_abegcweqnetpdlrh.jpg,a,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
dataloader_resnet = create_sun_dataloader(
    df=df_sun, img_dir='data/SUN/images/', transform=centre_crop,
    attr_cols=attr_cols, batch_size=32, num_workers=4
)

dataloader_clip = create_sun_dataloader(
    df=df_sun, img_dir='data/SUN/images/', transform=clip_preprocess,
    attr_cols=attr_cols, batch_size=64, num_workers=4)

In [22]:
def extract_clip_features(dataloader, model):
    """Przepuszcza obrazy przez CLIP i zwraca znormalizowane wektory."""
    all_features = []
    all_labels = []
    all_paths = []
    
    #model.eval()
    with torch.no_grad():
        for imgs, labels, paths in tqdm(dataloader, desc="Ekstrakcja cech CLIP"):
            features = model.encode_image(imgs.to(DEVICE))
            features = features / features.norm(dim=-1, keepdim=True)
            
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
            all_paths.extend(paths)
            
    return np.vstack(all_features), np.vstack(all_labels), all_paths


features_clip, labels_clip, paths_clip = extract_clip_features(dataloader_clip, clip_model)
df_labels = pd.DataFrame(labels_clip, columns=attr_cols)

Ekstrakcja cech CLIP: 100%|██████████| 225/225 [02:39<00:00,  1.41it/s]


In [23]:
def extract_resnet_features(dataloader, model):
    all_features = []
    
    with torch.no_grad():
        for imgs, labels, paths in tqdm(dataloader, desc="Downloading ResNet features"):
            features = model(imgs.to(DEVICE))
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            
    return np.vstack(all_features)

print("Downloading ResNet features...")
features_resnet = extract_resnet_features(dataloader_resnet, conv_model)

print(f"Shape of CLIP features: {features_clip.shape}")
print(f"Shape of ResNet features: {features_resnet.shape}")

Shape of CLIP features: (14340, 512)
Shape of ResNet features: (14340, 512)


In [24]:
indices = np.arange(len(df_sun))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

df_train = df_labels.iloc[train_idx].reset_index(drop=True)
df_test  = df_labels.iloc[test_idx].reset_index(drop=True)

X_train_clip = features_clip[train_idx]
X_test_clip  = features_clip[test_idx]

X_train_resnet = features_resnet[train_idx]
X_test_resnet  = features_resnet[test_idx]

print(f"Training set: {len(train_idx)} samples")
print(f"Test set: {len(test_idx)} samples")

def calculate_standard_cav(features_matrix, pos_indices, neg_indices):
    mean_concept = np.mean(features_matrix[pos_indices], axis=0)
    mean_background = np.mean(features_matrix[neg_indices], axis=0)
    
    cav = mean_concept - mean_background
    norm = np.linalg.norm(cav)
    if norm > 1e-8:
        return cav / norm
    raise ValueError('Norma CAV jest bliska zeru.')

Training set: 11472 samples
Test set: 2868 samples


In [25]:
hierarchies_to_test = [
    # empirical
    ('research', 'enclosed area'),
    ('camping', 'open area'),
    ('medical activity', 'enclosed area'),
    ('gaming', 'enclosed area'),
    ('sand', 'natural light'),
    ('carpet', 'enclosed area'),

    # sematical
    ('ocean', 'open area'),
    ('conducting business', 'working'),
    ('pavement', 'asphalt'),
    ('climbing', 'rock/stone'), 
    ('rusty', 'aged/ worn'),  
    ('ice', 'cold')
]

def evaluate_cav_methods(X_train, X_test, df_train, df_test, pairs, model_name):
    print("\n" + "="*80)
    print(f"EWALUATION FOR MODEL: {model_name.upper()}")
    print("="*80)
    print(f"{'SUBCONCEPT':<18} | {'PARENT':<15} | {'AUC STD CAV':<15} | {'AUC FILTERED CAV':<16}")
    print("-" * 80)
    
    results = []
    
    for child, parent in pairs:
        # --- TRENING ---
        # 1. Indeksy dla standardowego CAV (cały zbiór treningowy)
        child_pos_train = df_train.index[df_train[child] == 1.0].to_numpy()
        child_neg_train = df_train.index[df_train[child] == 0.0].to_numpy()
        
        # 2. Indeksy dla Filtered CAV (pojęcie rodzica musi istnieć)
        parent_pos_train = df_train.index[df_train[parent] == 1.0].to_numpy()
        
        # Zabezpieczenie przed brakiem danych
        if len(child_pos_train) == 0 or len(parent_pos_train) == 0:
            continue
            
        try:
            # Trenujemy oba wektory
            cav_std = calculate_standard_cav(X_train, child_pos_train, child_neg_train)
            cav_filtered = calculate_filtered_cav(X_train, child_pos_train, child_neg_train, parent_pos_train)
        except ValueError:
            continue # Pomijamy jeśli brakuje danych w przecięciu (Filtered)
            
            
        # --- TESTOWANIE (TWARDE NEGATYWY) ---
        # Filtrujemy zbiór testowy do obrazów, gdzie WYSTĘPUJE pojęcie rodzica
        hard_mask_test = df_test[parent] == 1.0
        
        y_true_hard = df_test[child][hard_mask_test].values
        X_hard = X_test[hard_mask_test]
        
        # Sprawdzamy czy mamy pozytywy i negatywy do wyliczenia AUC
        if len(np.unique(y_true_hard)) > 1:
            # Przewidywania (podobieństwo kosinusowe)
            scores_std = cosine_similarity(X_hard, cav_std.reshape(1, -1)).flatten()
            scores_filtered = cosine_similarity(X_hard, cav_filtered.reshape(1, -1)).flatten()
            
            auc_std = roc_auc_score(y_true_hard, scores_std)
            auc_filtered = roc_auc_score(y_true_hard, scores_filtered)
            diff = auc_filtered - auc_std


            results.append({
                'child': child, 'parent': parent, 
                'auc_std': auc_std, 
                'auc_filtered': auc_filtered,
                'auc_diff': diff
            })
            
            znak = "+++" if auc_filtered > auc_std else "---"
            print(f"{child:<18} | {parent:<15} | {auc_std:<15.4f} | {auc_filtered:<15.4f} {znak} | Diff: {diff:+.4f}")
        else:
            print(f"{child:<18} | {parent:<15} | {'---':<15} | {'--- (brak podziału w teście)'}")

    return pd.DataFrame(results)

# Uruchamiamy eksperyment
df_results_clip = evaluate_cav_methods(
    X_train_clip, X_test_clip, df_train, df_test, hierarchies_to_test, "CLIP (ViT-B/32)"
)

df_results_resnet = evaluate_cav_methods(
    X_train_resnet, X_test_resnet, df_train, df_test, hierarchies_to_test, "ResNet18 (Places365)"
)


EWALUATION FOR MODEL: CLIP (VIT-B/32)
SUBCONCEPT         | PARENT          | AUC STD CAV     | AUC FILTERED CAV
--------------------------------------------------------------------------------
research           | enclosed area   | 0.9186          | 0.9136          --- | Diff: -0.0050
camping            | open area       | 0.8798          | 0.8935          +++ | Diff: +0.0137
medical activity   | enclosed area   | 0.9739          | 0.9701          --- | Diff: -0.0039
gaming             | enclosed area   | 0.9506          | 0.9589          +++ | Diff: +0.0083
sand               | natural light   | 0.8858          | 0.9015          +++ | Diff: +0.0158
carpet             | enclosed area   | 0.7604          | 0.7628          +++ | Diff: +0.0024
ocean              | open area       | 0.9600          | 0.9627          +++ | Diff: +0.0027
conducting business | working         | 0.7745          | 0.6962          --- | Diff: -0.0783
pavement           | asphalt         | 0.6000          | 0.59

### ASD

In [26]:
indices = np.arange(len(df_sun))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

df_train = df_labels.iloc[train_idx].reset_index(drop=True)
df_test  = df_labels.iloc[test_idx].reset_index(drop=True)

# Podział cech CLIP
X_train_clip = features_clip[train_idx]
X_test_clip  = features_clip[test_idx]

# Podział cech ResNet
X_train_resnet = features_resnet[train_idx]
X_test_resnet  = features_resnet[test_idx]

print(f"Zbiór treningowy: {len(train_idx)} próbek")
print(f"Zbiór testowy: {len(test_idx)} próbek")

all_pairs = [
    # empirical
    ('research', 'enclosed area'),
    ('camping', 'open area'),
    ('medical activity', 'enclosed area'),
    ('gaming', 'enclosed area'),
    ('sand', 'natural light'),
    ('carpet', 'enclosed area'),

    # sematical
    ('ocean', 'open area'),
    ('conducting business', 'working'),
    ('pavement', 'asphalt'),
    ('climbing', 'rock/stone'), 
    ('rusty', 'aged/ worn'),  
    ('ice', 'cold')
]

child_to_parent = {a: b for a, b in all_pairs}

Zbiór treningowy: 11472 próbek
Zbiór testowy: 2868 próbek


In [29]:
def run_evaluation(X_train_features, X_test_features, model_name):
    print("\n" + "="*125)
    print(f" EWALUATION FOR MODEL: {model_name.upper()}")
    print("="*125)
    print(f"{'CONCEPT (CHILD)':<22} | {'PARENT':<22} | {'AUC (STANDARD)':<15} | {'AUC (FILTERED)':<15} | {'DIFFERENCE'}")
    print("-" * 125)
    
    # Zmienna do przechowywania wyników przed ich wyświetleniem
    results = []
    
    for concept in child_to_parent.keys():
        parent_concept = child_to_parent[concept]
        
        # 1. Indeksy dla standardowego CAV (cały zbiór treningowy)
        idx_pos_train = df_train.index[df_train[concept] == 1.0].tolist()
        idx_neg_train = df_train.index[df_train[concept] == 0.0].tolist()
        
        # 2. Indeksy dla Filtered CAV (pojęcie rodzica musi istnieć)
        idx_parent_train = df_train.index[df_train[parent_concept] == 1.0].tolist()

        if len(idx_pos_train) == 0 or len(idx_parent_train) == 0:
            continue
            
        try:
            # Trenujemy oba wektory na danych treningowych
            cav_std = calculate_centroid_cav(X_train_features, idx_pos_train, idx_neg_train)
            cav_filtered = calculate_filtered_cav(X_train_features, idx_pos_train, idx_neg_train, idx_parent_train)
        except ValueError:
            continue
            
        # 3. Testowanie (TWARDE NEGATYWY)
        # Filtrujemy zbiór testowy do obrazów, gdzie WYSTĘPUJE pojęcie rodzica
        hard_mask = (df_test[parent_concept] == 1.0).values
        
        y_test_hard = df_test[concept].values[hard_mask]
        X_test_hard = X_test_features[hard_mask]
        
        # Obliczenie AUC, jeśli mamy w zbiorze testowym przynajmniej jeden przypadek pozytywny i negatywny
        if len(np.unique(y_test_hard)) > 1:
            scores_hard_std = cosine_similarity(X_test_hard, cav_std.reshape(1, -1)).flatten()
            scores_hard_filtered = cosine_similarity(X_test_hard, cav_filtered.reshape(1, -1)).flatten()
            
            auc_std = roc_auc_score(y_test_hard, scores_hard_std)
            auc_filtered = roc_auc_score(y_test_hard, scores_hard_filtered)
            
            # Różnica klasycznie wg Twojego kodu
            auc_diff = auc_std - auc_filtered
             
            # Dodajemy pomyślnie policzone wyniki
            results.append({
                'concept': concept,
                'parent': parent_concept,
                'auc_std': auc_std,
                'auc_filtered': auc_filtered,
                'auc_diff': auc_diff,
                'valid': True
            })
        else:
            # Dodajemy przypadek ze zbyt małą próbą badawczą
            results.append({
                'concept': concept,
                'parent': parent_concept,
                'valid': False
            })
            
    # Podział na wyniki poprawne i niewystarczające
    valid_results = [r for r in results if r['valid']]
    invalid_results = [r for r in results if not r['valid']]
    
    # Sortowanie poprawnych wyników
    # Sortujemy malejąco według (auc_filtered - auc_std), co sprawia, 
    # że największa poprawa dzięki Filtered CAV jest na samej górze.
    valid_results.sort(key=lambda x: x['auc_filtered'] - x['auc_std'], reverse=True)
    
    # Wypisywanie posortowanych wyników
    for res in valid_results:
        print(f"{res['concept']:<22} | {res['parent']:<22} | {res['auc_std']:<15.3f} | {res['auc_filtered']:<15.3f} | Diff: {res['auc_diff']:+.3f}")
        
    # Wypisywanie błędów po poprawnych wynikach
    for res in invalid_results:
        print(f"{res['concept']:<22} | {res['parent']:<22} | {'---':<15} | {'---':<15} | Zbyt mała próba testowa")

In [30]:
run_evaluation(X_train_clip, X_test_clip, "CLIP (ViT-B/32)")
run_evaluation(X_train_resnet, X_test_resnet, "ResNet18 (Places365)")


 EWALUATION FOR MODEL: CLIP (VIT-B/32)
CONCEPT (CHILD)        | PARENT                 | AUC (STANDARD)  | AUC (FILTERED)  | DIFFERENCE
-----------------------------------------------------------------------------------------------------------------------------
sand                   | natural light          | 0.886           | 0.902           | Diff: -0.016
camping                | open area              | 0.880           | 0.894           | Diff: -0.014
gaming                 | enclosed area          | 0.951           | 0.959           | Diff: -0.008
climbing               | rock/stone             | 0.838           | 0.844           | Diff: -0.006
ocean                  | open area              | 0.960           | 0.963           | Diff: -0.003
carpet                 | enclosed area          | 0.760           | 0.763           | Diff: -0.002
pavement               | asphalt                | 0.600           | 0.597           | Diff: +0.003
medical activity       | enclosed area      